In [ ]:
import json, time, os, re
import pandas as pd
import shutil
import io
from PIL import Image

from enum import Enum
from typing import Any, List, Dict, Optional, Union

from src.file_service import read_file, write_file
from src.scryfall_http_service import ScryfallHttpService
from src.scryfall_data_service import ScryfallDataService
from src.cache_service import CacheService, CacheType
from src.image_service import ImageService

pd.set_option('display.max_columns', None)

SCRYFALL_BASE_URL = "https://api.scryfall.com/"
CARDS_COLUMNS = [
    'name',
    'oracle_id',
    # 'image_uris',
    'image_uri',
    'mana_cost',
    'cmc',
    'type_line',
    'oracle_text',
    'toughness',
    'colors',
    'color_identity',
    'keywords',
    # 'legalities',
    # 'set',
    # 'set_name',
    # 'scryfall_set_uri',
    # 'rulings_uri',
    # 'prints_search_uri',
    # 'flavor_text',
    'edhrec_rank',
    # 'prices',
    'price_usd',
]
os.getcwd()

'c:\\Users\\tallo\\Documents\\programming_projects\\mtg-coding'

# Fetching Images

In [ ]:
json_cache_service = CacheService("./cache/json/", cache_type=CacheType.JSON)
png_cache_service = CacheService("./cache/png/", cache_type=CacheType.PNG)
scryfall_http_client = ScryfallHttpService()

sfdh = ScryfallDataService(SCRYFALL_BASE_URL, json_cache_service, png_cache_service, scryfall_http_client)

In [ ]:
cards_df = sfdh.get_oracle_cards()
cards_df.shape

(38233, 90)

In [10]:
COMMANDERS_LIST_FPATH = "phils_commanders.txt"

def load_list_of_cards(fpath: str) -> List[str]:
    return sorted([x.strip() for x in read_file(fpath).split("\n") if x.strip()])

for f in os.listdir("./data/lists/"):
    print(f)
    cards = load_list_of_cards(os.path.join("./data/lists/", f))
    loaded_cards_df = cards_df[cards_df['name'].isin(cards)]
    names_to_uri_dict = {r['name']: r['image_uri'] for r in loaded_cards_df.to_dict('records')}
    cache_card_images(names_to_uri_dict, scryfall_http_client, png_cache_handler)
    save_images_as_grid(
        images=get_images_from_cache(sorted(loaded_cards_df.name.tolist()), png_cache_handler), 
        columns=4, 
        output_path=os.path.join("./data/grids/", f"{f.split('.')[0]}.png"),
    )

kaleys_commanders.txt
phils_commanders.txt
phils_top_commanders.txt
